# Green SM Food Recommendation — Kaggle full v1

This notebook trains the frozen-Foundation FoodRanker from one opaque linked full-v1 Food runtime payload, using the selected full-v1 Foundation checkpoint, then writes checksum-bound, label-free validation and test predictions. Cell 2 decodes and safely extracts the payload with visible progress, so Kaggle does not need to mount 24,000 small input files before the kernel starts. Enable a GPU and Internet before running. It deliberately does not use the control corpus or Ads data.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/repo')
if REPO.exists():
    raise RuntimeError('Fresh Kaggle session required: /kaggle/working/repo already exists.')

subprocess.run([
    'git', 'clone', '--branch', 'feat/foundation-encoder',
    'https://github.com/huongiang171/green-sm-foundation-pretraining.git',
    str(REPO),
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
print(subprocess.check_output(['git', 'log', '-1', '--oneline'], cwd=REPO, text=True).strip())

def run_food_cli(command: list[str], *, label: str) -> None:
    """Forward every child-process line to the notebook output immediately."""
    print(f'{label}: launch', flush=True)
    process = subprocess.Popen(
        command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


In [ ]:
from pathlib import Path
import json
import base64
import tarfile
import torch

INPUT_ROOT = Path('/kaggle/input')

def kaggle_dataset_roots() -> list[Path]:
    # Kaggle normally mounts /kaggle/input/<slug>. This account has an
    # additional /datasets/<owner>/<slug> layer. Bound the search depth
    # instead of recursively walking every mounted Parquet file.
    candidates = [
        INPUT_ROOT,
        *INPUT_ROOT.iterdir(),
        *INPUT_ROOT.glob('*/*'),
        *INPUT_ROOT.glob('*/*/*'),
    ]
    return [path for path in candidates if path.is_dir()]

def find_exact_root(name: str, required_relative: str) -> Path:
    matches = [
        dataset_root / name
        for dataset_root in kaggle_dataset_roots()
        if (dataset_root / name / required_relative).is_file()
    ]
    assert len(matches) == 1, (
        f'Expected one {name!r} root, found {matches}. '
        f'Input mounts: {[str(path) for path in kaggle_dataset_roots()]}'
    )
    return matches[0]

def find_exact_file(name: str) -> Path:
    matches = [dataset_root / name for dataset_root in kaggle_dataset_roots() if (dataset_root / name).is_file()]
    assert len(matches) == 1, (
        f'Expected one archive {name!r}, found {matches}. '
        f'Input mounts: {[str(path) for path in kaggle_dataset_roots()]}'
    )
    return matches[0]

FOOD_RUNTIME_PAYLOAD = find_exact_file('green_sm_food_full_runtime_v1.tar.gz.b64')
CHECKPOINT_ROOT = find_exact_root('green_sm_full_v1_checkpoint', 'best.pt')
# Keep the 24k-file decoded runtime out of /kaggle/working so Kaggle
# only saves model artifacts and predictions when committing this notebook.
RUNTIME_ROOT = Path('/tmp/green_sm_food_runtime')
if RUNTIME_ROOT.exists():
    raise RuntimeError(f'Fresh Kaggle session required: {RUNTIME_ROOT} already exists.')
RUNTIME_ROOT.mkdir(parents=True)

DECODED_ARCHIVE = RUNTIME_ROOT / 'green_sm_food_full_runtime_v1.tar.gz'
print('Food runtime decode: start', flush=True)
DECODED_ARCHIVE.write_bytes(base64.b64decode(FOOD_RUNTIME_PAYLOAD.read_bytes(), validate=True))
print(f'Food runtime decode: complete ({DECODED_ARCHIVE.stat().st_size:,} bytes)', flush=True)

with tarfile.open(DECODED_ARCHIVE, mode='r:gz') as archive:
    members = archive.getmembers()
    print(f'Food runtime extraction: {len(members)} archive entries | start', flush=True)
    destination = RUNTIME_ROOT.resolve()
    progress_interval = max(1, len(members) // 10)
    for index, member in enumerate(members, start=1):
        member_path = (RUNTIME_ROOT / member.name).resolve()
        if not member_path.is_relative_to(destination) or not (member.isfile() or member.isdir()):
            raise RuntimeError(f'Unsafe Food runtime archive member: {member.name!r}')
        archive.extract(member, path=RUNTIME_ROOT)
        if index % progress_interval == 0 or index == len(members):
            print(f'Food runtime extraction: entry {index}/{len(members)}', flush=True)

LINKED_ROOT = RUNTIME_ROOT / 'green_sm_sim_linked_v1'

for required in (
    LINKED_ROOT / 'food_opportunities' / 'manifest.json',
    LINKED_ROOT / 'foundation' / 'manifest.json',
    CHECKPOINT_ROOT / 'best.pt',
    CHECKPOINT_ROOT / 'best_manifest.json',
    CHECKPOINT_ROOT / 'checkpoint_manifest.json',
    CHECKPOINT_ROOT / 'vocabulary.json',
):
    assert required.is_file(), f'Missing required artifact: {required}'

selection = json.loads((CHECKPOINT_ROOT / 'checkpoint_manifest.json').read_text())
assert selection['selected_kind'] == 'best', selection
assert selection['checkpoint'] == 'best.pt', selection

assert torch.cuda.is_available(), 'Enable a Kaggle GPU Accelerator before training.'
print('GPU:', torch.cuda.get_device_name(0))
print('Food runtime payload:', FOOD_RUNTIME_PAYLOAD)
print('Extracted linked corpus:', LINKED_ROOT)
print('Foundation checkpoint:', CHECKPOINT_ROOT / 'best.pt')
print('Input discovery uses bounded mount-depth search; no recursive scan.')


In [ ]:
import hashlib
import yaml

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

with (REPO / 'configs' / 'food_recommendation_full.yaml').open(encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

food_root = LINKED_ROOT / 'food_opportunities'
foundation_root = LINKED_ROOT / 'foundation'
run_dir = Path('/kaggle/working/artifacts/food/food-full-v1-r1')
if run_dir.exists():
    raise RuntimeError(f'Choose a new run_id or restart: {run_dir} already exists.')

config['data'].update({
    'data_version': 'green_sm_sim_full_v1',
    'opportunities_root': str(food_root),
    'manifest_path': str(food_root / 'manifest.json'),
    'foundation_root': str(foundation_root),
    'foundation_manifest_path': str(foundation_root / 'manifest.json'),
    'foundation_corpus': 'linked',
    'vocabulary_path': str(CHECKPOINT_ROOT / 'vocabulary.json'),
    'expected_food_manifest_sha256': sha256(food_root / 'manifest.json'),
    'expected_foundation_manifest_sha256': sha256(foundation_root / 'manifest.json'),
})
config['checkpoint'].update({
    'path': str(CHECKPOINT_ROOT / 'best.pt'),
    'manifest_path': str(CHECKPOINT_ROOT / 'best_manifest.json'),
    'expected_sha256': sha256(CHECKPOINT_ROOT / 'best.pt'),
})
config['ranker'].update({
    'run_dir': str(run_dir),
    # This is ignored by train; the real SHA is filled after train before predict.
    'expected_run_manifest_sha256': '0' * 64,
})
config['training'].update({'device': 'cuda', 'epochs': 20, 'batch_size': 16})

RUNTIME_CONFIG = Path('/kaggle/working/food_recommendation_full_v1.yaml')
with RUNTIME_CONFIG.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

print(RUNTIME_CONFIG.read_text(encoding='utf-8'))


In [ ]:
run_food_cli([
    sys.executable, '-u', '-m', 'src.downstream.food_recommendation',
    '--config', str(RUNTIME_CONFIG), '--mode', 'validate',
], label='Food handoff validation')
print('Food handoff validation passed.', flush=True)


In [ ]:
print('Food training starts now; live stage logs are forwarded line-by-line.', flush=True)
run_food_cli([
    sys.executable, '-u', '-m', 'src.downstream.food_recommendation',
    '--config', str(RUNTIME_CONFIG), '--mode', 'train',
], label='Food training')


In [ ]:
# Bind future prediction commands to exactly the Food run just trained.
with RUNTIME_CONFIG.open(encoding='utf-8') as handle:
    config = yaml.safe_load(handle)
config['ranker']['expected_run_manifest_sha256'] = sha256(run_dir / 'run_manifest.json')
with RUNTIME_CONFIG.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)
print('Food run manifest SHA:', config['ranker']['expected_run_manifest_sha256'])


In [ ]:
for split in ('validation', 'test'):
    run_food_cli([
        sys.executable, '-u', '-m', 'src.downstream.food_recommendation',
        '--config', str(RUNTIME_CONFIG), '--mode', 'predict', '--split', split,
    ], label=f'Food prediction ({split})')


In [ ]:
import json
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

metrics = [json.loads(line) for line in (run_dir / 'train_metrics.jsonl').read_text().splitlines()]
plt.plot([row['epoch'] for row in metrics], [row['train_listwise_loss'] for row in metrics], label='train')
plt.plot([row['epoch'] for row in metrics], [row['validation_listwise_loss'] for row in metrics], label='validation')
plt.xlabel('epoch')
plt.ylabel('multi-positive listwise loss')
plt.legend()
plt.show()

for split in ('validation', 'test'):
    folder = run_dir / 'predictions' / split
    report = json.loads((folder / f'validation_report.{split}.json').read_text())
    table = pq.read_table(folder / f'predictions.{split}.parquet')
    print(split, report, 'rows=', table.num_rows)
    display(table.to_pandas().head())


In [ ]:
import shutil

archive = shutil.make_archive('/kaggle/working/green_sm_food_full_v1_r1', 'zip', root_dir=run_dir)
print('Download this Kaggle output:', archive)
